# 3. 图的状态（State）管理
## 3.1. 状态定义

从教程来看,State 是整个图共享的数据快照。节点读取它，并返回对它的局部更新

> 官方推荐了三种定义Schema的方式：TypedDict、dataclass、Pydantic

这种定义的方式和 langchain的结构化输出一模一样。

### 3.1.1 TypedDict写法

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from operator import add

class OverAllState(TypedDict):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))

{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


### 3.1.2 dataclass

属性调用方式由`['字段名']`变为`.字段名`。

In [2]:
from langgraph.graph import StateGraph, START, END
from typing import  Annotated
from operator import add
from dataclasses import dataclass

@dataclass
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state["cur_id"]
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))


TypeError: 'OverAllState' object is not subscriptable

如果仍然采用 字典的形式，会出现 `'OverAllState' object is not subscriptable` 错误, 指“不支持下标/索引操作”

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import  Annotated
from operator import add
from dataclasses import dataclass

@dataclass
class OverAllState:
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {# 自动转换为 OverAllState
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))


{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


### 3.1.3. Pydantic

Pydantic 是一个 Python 的数据验证库，它提供了数据类型和验证的定义方式，可以用于数据的校验和转换.Pydantic模型的字段访问方式和dataclass相同。

In [4]:
from langgraph.graph import StateGraph, START, END
from typing import  Annotated
from operator import add
from pydantic import BaseModel

class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_2 运行完毕"],
        "cur_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))


{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1, node_2'}


### 3.1.4 校验行为


#### 3.1.4.1 节点返回字段不匹配
图节点返回的是对于状态的更新，如果返回字段和状态字段不匹配，上述三种Schema定义方式的行为是统一的：状态更新会被忽略。

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import  Annotated
from operator import add
from pydantic import BaseModel

class OverAllState(BaseModel):
    logs: Annotated[list[str], add]
    cur_id: str

def node_1(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_1 运行完毕"],
        "cur_id": pre_id + ", node_1"
    }

def node_2(state: OverAllState) -> OverAllState:
    pre_id = state.cur_id
    return {
        "logs": ["node_2 运行完毕"],
        "s_id": pre_id + ", node_2"
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_edge(START, "node_1")
builder.add_edge("node_1", "node_2")
builder.add_edge("node_2", END)

graph = builder.compile()

print(graph.invoke({"cur_id": "start"}))


{'logs': ['node_1 运行完毕', 'node_2 运行完毕'], 'cur_id': 'start, node_1'}


#### 补充实验：输入字段不匹配（课程 3.1.4.1）与节点返回错误类型

运行前先预测：三种 Schema 各自会抛什么异常？哪种会把错误类型的值原样传给下游节点？

In [ ]:
from typing import TypedDict
from dataclasses import dataclass
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END

class TDState(TypedDict):
    cur_id: str

@dataclass
class DCState:
    cur_id: str

class PDState(BaseModel):
    cur_id: str

def read_cur_id(state):
    return state["cur_id"] if isinstance(state, dict) else state.cur_id

def build(schema, first_node):
    def node_2(state):
        value = read_cur_id(state)
        print(f"    node_2 收到 cur_id={value!r}，类型 {type(value).__name__}")
        return {}
    builder = StateGraph(schema)
    builder.add_node("node_1", first_node)
    builder.add_node("node_2", node_2)
    builder.add_edge(START, "node_1")
    builder.add_edge("node_1", "node_2")
    builder.add_edge("node_2", END)
    return builder.compile()

def try_invoke(label, graph, inputs):
    try:
        print(f"{label}: 结果 {graph.invoke(inputs)}")
    except Exception as e:
        print(f"{label}: {type(e).__name__}: {str(e).splitlines()[0]}")

print("实验 A：输入字段名不匹配（课程 3.1.4.1）")
for schema in (TDState, DCState, PDState):
    graph = build(schema, lambda state: {"cur_id": read_cur_id(state) + ",node_1"})
    try_invoke(f"  {schema.__name__}", graph, {"wrong": "start"})  # ← 改动：传入不存在的字段

print("实验 B：节点返回错误类型")
for schema in (TDState, DCState, PDState):
    graph = build(schema, lambda state: {"cur_id": 123})  # ← 改动：cur_id 应为 str
    try_invoke(f"  {schema.__name__}", graph, {"cur_id": "start"})

### 3.1.5 推荐使用
貌似使用`TypedDict` 是官方的做法。可能langgraph 在运行时，对数据格式要求并不严格?

# 结论

1. State Schema 有三种定义方式：TypedDict、dataclass、Pydantic。写法和 LangChain 结构化输出相同，但作为图状态时的校验行为不同（见第 4 条）。
2. **读取字段的方式随 Schema 变化**：TypedDict 用 `state["cur_id"]`；dataclass 和 Pydantic 用 `state.cur_id`。混用会报 `'OverAllState' object is not subscriptable`（3.1.2 的实验）。
3. **节点返回值的格式不随 Schema 变化**：三种写法下，节点都返回 dict 形式的局部更新。LangGraph 把它按字段拆开，交给对应字段的 reducer 合并，没有 reducer 的字段直接覆盖。返回 Schema 中不存在的字段（如 `s_id`）会被静默忽略，不报错。
4. **运行时校验**（见上方补充实验 A、B）：

   | 情况 | TypedDict | dataclass | Pydantic |
   |---|---|---|---|
   | 输入字段名不匹配 | 不校验，未知字段被丢弃；节点读取缺失字段时，由**节点代码**抛 `KeyError` | 构造实例时抛 `TypeError`（缺少必填字段） | 抛 `ValidationError` |
   | 节点返回错误类型 | 不校验，下游节点拿到 `int` | 不校验，下游节点拿到 `int` | 构造下一个节点的输入时抛 `ValidationError` |

   Pydantic 只在**构造节点输入**时校验。如果最后一个节点返回了错误类型，后面已经没有节点需要构造输入，错误的值会原样出现在最终结果里。
5. **推荐 TypedDict**：节点返回的本来就是部分更新的 dict，TypedDict 与之最贴合，而且没有校验开销。需要在运行时校验数据时（例如输入来自外部、不可信），再考虑 Pydantic。

# 问题

1. 3.1.5 的疑问：LangGraph 在运行时对数据格式要求不严格吗？

   结论：对 TypedDict 和 dataclass 来说是的。类型注解只是给 IDE 和读者看的提示，运行时不检查类型；dataclass 只检查必填字段是否齐全。只有 Pydantic 会校验类型，而且只在构造节点输入时校验（见结论第 4 条）。